# Model Context Protocol (MCP)

## 1. Introduction

### 1.1 Overview of MCP

MCP (Model Context Protocol) is an open-source standard that connects AI apps to external systems — data sources (files, databases), tools (search, calculators), and workflows (prompts).

Instead of every app rebuilding tool integrations from scratch, MCP moves tool definitions and execution into dedicated **MCP servers** that any AI app can plug into.

> **Think of it as USB-C for AI apps** — one standard, plug anything into anything.

---

### 1.2 MCP Clients and Servers
![描述文字](https://everpath-course-content.s3-accelerate.amazonaws.com/instructor%2Fa46l9irobhg0f5webscixp0bs%2Fpublic%2F1749849216%2F09_-_001_-_Introducing_MCP_01.1749849216723.png)

**Client** serves as the communication bridge between your server and MCP servers. Think of it as your access point to all the tools that an MCP server provides.  When you want to connect your AI app with outside MCP server, you build an MCP client to connect your application to an MCP server.

**Servers** host resources, tools, and prompts. **Clients** use servers.

A server can expose three things:

| Component | What it does |
|---|---|
| **Tools** | Implemented on a server; extend an LLM's capabilities (callable functions) |
| **Resources** | Deliver any kind of document or information to clients (readable data) |
| **Prompts** | Provide clients with pre-optimized prompt templates |


---

### 1.3 Message Types

MCP defines a few message types that flow between client and server. The two most important ones for tool use:

| Request | Result | Purpose |
|---|---|---|
| `ListToolsRequest` | `ListToolsResult` | Client asks: *"What tools do you provide?"* → Server returns the list of available tools |
| `CallToolRequest` | `CallToolResult` | Client asks: *"Run this specific tool with these arguments"* → Server returns the execution result |

---

### 1.4 The Complete Workflow

<img src="https://everpath-course-content.s3-accelerate.amazonaws.com/instructor%2Fa46l9irobhg0f5webscixp0bs%2Fpublic%2F1749849232%2F09_-_002_-_MCP_Clients_19.1749849231568.png" width="700">


End-to-end flow when a user asks a question that requires an external tool (e.g., querying GitHub):

1. **User Query** — User submits their question to your server.

2. **Tool Discovery** — Your server needs to know what tools are available to send to Claude.

3. **List Tools Exchange** — Your server asks the MCP client for the list of available tools.

4. **MCP Communication** — MCP client sends `ListToolsRequest` → MCP server; MCP server returns `ListToolsResult` → MCP client.

5. **Claude Request** — Your server sends `[user query + available tools]` to Claude.

6. **Tool Use Decision** — Claude decides it needs to call a tool to answer the question.

7. **Tool Execution Request** — Your server asks the MCP client to run the tool Claude specified.

8. **External API Call** — MCP client sends `CallToolRequest` → MCP server; MCP server makes the actual GitHub API call.

9. **Results Flow Back** — GitHub responds with repository data → MCP server wraps it as `CallToolResult` → flows back to your server.

10. **Tool Result to Claude** — Your server sends the tool results back to Claude.

11. **Final Response** — Claude formulates a final answer using the repository data.

12. **User Gets Answer** — Your server delivers Claude's response back to the user.

---

> 💡 **Key insight**: The MCP client/server pair handles **steps 3–4** (discovery) and **steps 7–9** (execution). Everything else is the same tool-use loop you already know from the Anthropic SDK — MCP just outsources *"what tools exist"* and *"actually run the tool"* to a separate process.
---

### 1.5 Common Questions

**Q: Who authors MCP servers?**

Anyone. Often, service providers release their own official implementations — e.g., AWS might publish an official MCP server exposing tools for their services.

**Q: How is this different from calling APIs directly?**

MCP servers come with **tool schemas and functions already defined for you**. Calling an API directly means writing those tool definitions yourself — MCP saves that implementation work.

**Q: Isn't MCP just the same as tool use?**

Common misconception — they're **complementary, not the same**:

- **Tool use** = how Claude *calls* tools (the mechanism)
- **MCP servers** = where the tool definitions and implementations *come from* (the source)

The key difference is **who does the work**: with MCP, someone else has already implemented the tools for you.

**Bottom line:** Instead of maintaining a complex web of integrations yourself, MCP servers handle the heavy lifting of connecting to external services.

---

## 2. Hands-on with MCP Servers

### 2.1 Project setup
see code details in subfolder 10a_cli_MCPproject

---
### 2.2 Defining tools with MCP
Building an MCP server becomes much simpler when you use the official Python SDK. Instead of writing complex JSON schemas by hand, you can define tools with decorators and let the SDK handle the heavy lifting.

### Mental Model — "Opening a Shop"

| Code | Analogy |
|---|---|
| `from mcp.server.fastmcp import FastMCP` | **Stocking up** — buying a cash register |
| `mcp = FastMCP("DocumentMCP", ...)` | **Opening up** — hanging the storefront sign, placing the register on the counter |
| `@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()` (later) | **Adding the menu** — registering what the shop offers |
| `mcp.run()` (final line) | **Open for business** — letting customers in |

1. Setting Up the MCP Server with Python MCP SDK + storing documents

2. Tool Definition with Decorators
    
    (1) Creating a Document Reader Tool

    (2) Building a Document Editor Tool
---
### 2.3 The server inspector
When building MCP servers, you need a way to test your functionality without connecting to a full application. The Python MCP SDK includes a built-in browser-based inspector that lets you debug and test your server in real-time.

1. Starting the Inspector
2. Using the Inspector Interface
3. Testing Your Tools

---

## 3. Connecting with MCP Clients

### 3.1 Implementing a client
1. The MCP client consists of two main components:

    (1) MCP Client - A custom class we create to make using the session easier

    (2) Client Session - The actual connection to the server (part of the MCP Python SDK)

2. Implementing Core Client Functions

    (1) List Tools Function

    (2) Call Tools Function

3. Testing the client
---

### 3.2 Defining Resources (MCP Server Side)
1. Understanding Resources

    Resources in MCP servers allow you to expose data to clients, similar to GET request handlers in a typical HTTP server. They're perfect for scenarios where you need to fetch information rather than perform actions.

    Let's say you want to build a document mention feature where users can type @document_name to reference files. This requires two operations:

    Getting a list of all available documents (for autocomplete)
    Fetching the contents of a specific document (when mentioned)


2. How resources works

    <img src="https://everpath-course-content.s3-accelerate.amazonaws.com/instructor%2Fa46l9irobhg0f5webscixp0bs%2Fpublic%2F1749849288%2F09_-_007_-_Defining_Resources_05.1749849288268.png" width="700">


3. Types of Resources

    There are two types of resources:

    (1) **Direct Resources**
    Direct resources have static URIs that never change. They're perfect for operations that don't need parameters.

    (2) **Templated Resources**
    Templated resources include parameters in their URIs. The Python SDK automatically parses these parameters and passes them as keyword arguments to your function.


4. Implementation details

    Resources can return any type of data - strings, JSON, binary data, etc. Use the mime_type parameter to give clients a hint about what kind of data you're returning:

    "application/json" for structured data

    "text/plain" for plain text

    "application/pdf" for binary files

    The MCP Python SDK automatically serializes your return values. You don't need to manually convert objects to JSON strings - just return the data structure and let the SDK handle serialization.


---

### 3.3 Accessing Resources (MCP Client Side)

1. Implementing Resources Reading --> define read_resource function

2. Understanding the Response Structure
    
    When you request a resource, the server returns a result with a contents list. We access the first element since we typically only need one resource at a time. The response includes:

    The actual content (text or data)
        A MIME type that tells us how to parse the content
        Other metadata about the resource

3. Content Type Handling

    The function checks the MIME type to determine how to process the content:

    If it's application/json, parse the text as JSON and return the parsed object
    Otherwise, return the raw text content
    This approach handles both structured data (like JSON) and plain text documents seamlessly.


4. Testing Resource Access

    Once implemented, you can test the resource functionality through your CLI application. When you type "@" followed by a resource name, the system will:

    Show available resources in an autocomplete list
    Let you select a resource using arrow keys and space
    Include the resource content directly in your prompt
    Send everything to the AI model without requiring additional tool calls
    This creates a much smoother user experience compared to having the AI model make separate tool calls to access document contents. The resource content becomes part of the initial context, allowing for immediate responses about the data.

---

### 3.4 Defining Prompts (MCP Server Side)

1. why prompts?

    Prompts in MCP servers let you define pre-built, high-quality instructions that clients can use instead of writing their own prompts from scratch. Think of them as carefully crafted templates that give better results than what users might come up with on their own.

    Let's implement a practical example: a format command that converts documents to markdown. Users will type /format doc_id and get back a professionally formatted markdown version of their document.

    The workflow looks like this:

    User types / to see available commands
    They select format and specify a document ID
    Claude uses your pre-built prompt to read and reformat the document
    The result is clean markdown with proper headers, lists, and formatting

2. Defining Prompts -- implementation code

3. Testing Your Prompts
---

### 3.5 Prompts in the client (MCP Client Side)

1. list_prompts() + get_prompt()
---